## Clip datasets to each Parish council
Author: Lara Spearman <br>
Date: 01/09/2025 <br>
Purpose: This script only needs re-run when datasets are updated. It outputs csv for each dataset split by parish council. 

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import json

In [ ]:
# Get current working directory
cwd = Path.cwd()

# Set location of data
location_data = "C:/Users/LaraS/Wiltshire Wildlife Trust/WSBRC Data - Documents/Services/Neighbourhood_Planning/_NeigbourhoodPlansPython/NeigbourhoodPlansPython.gdb"

In [3]:
# Read in parish council boundaries
gdf = gpd.read_file(location_data, layer="OS_BL_Parish_Wilts")
gdf = gdf.to_crs("EPSG:27700")
gdf['AreaPC'] = gdf['Hectares']

In [4]:
# Import file information to read in
with open('file_info_gdb.json') as f:
    file_info_shp = json.load(f)
    print(file_info_shp)

{'Landscape Character Areas': {'file_name': '', 'source': 'Wiltshire Council', 'group': 'LCA', 'group_field': 'Type_name', 'groupColumnName': 'Area', 'groupColumnValue': False, 'unit': 'Ha', 'unit_type': 'AREA', 'area_col_name': 'AREA', 'about': 'Landscape Character Areas represent an area of distinct and recognisable character at the national scale. Their boundaries follow natural lines in the landscape, not county or district boundaries. This makes them a good framework for decision-making and planning for future change. ', 'layer_name': 'LCA_County'}, 'Bedrock': {'file_name': '', 'source': 'British Geological Survey', 'group': 'Geology', 'group_field': 'LEX_RCS_D', 'groupColumnName': 'Bedrock', 'groupColumnValue': False, 'unit': 'Ha', 'unit_type': 'AREA', 'area_col_name': 'AREA', 'about': 'test', 'layer_name': 'Bedrock'}, 'Faultlines': {'file_name': '', 'source': 'British Geological Survey', 'group': 'Geology', 'group_field': 'CATEGORY', 'groupColumnName': 'Faultline', 'groupColumnV

In [5]:
# Create empty dataframe to fill
df_all_shp = pd.DataFrame({'Name': [], 'value': [],'groupColumnValue': [],'groupColumnName': [],'mapGroup': [], "unitName":[],"unit":[],"datasetName":[], "source":[]})

# Loop through each dataset listed in dictionary
for file in file_info_shp.keys():

    gdf_data = gpd.read_file(location_data + file_info_shp[file]["file_name"], layer = file_info_shp[file]["layer_name"])
    
    gdf_data = gdf_data.to_crs("EPSG:27700")
    print(file)

    # Create empty dataframe to fill for each PC
    df_local_shp = pd.DataFrame({'Name': [], 'value': [],'groupColumnValue': [],'groupColumnName': [],'mapGroup': [], "unitName":[],"unit":[],"datasetName":[], "source":[]})

     # Get stats for each PC
    for pc in gdf['Name'].unique():
        # Filter parish council dataframe to relevant PC
        gdf_local = gdf[gdf.Name == pc]
        # Clip data to PC
        gdf_clipped = gdf_data.clip(gdf_local)

        # If dataset is a count, just need to add all occurences
        if file_info_shp[file]["unit_type"].lower()=="count":
            gdf_dissolved = gdf_clipped[[file_info_shp[file]["count_col_name"],"geometry"]].dissolve( aggfunc="count")
            gdf_dissolved['value'] = gdf_dissolved[file_info_shp[file]["count_col_name"]]
        else:
            # If dataset is split by groups, need to summarise for each group
            if file_info_shp[file]["group_field"]:
                gdf_dissolved = gdf_clipped[[file_info_shp[file]["group_field"],"geometry"]].dissolve(by = file_info_shp[file]["group_field"], aggfunc="sum").reset_index()
            else:
                # If no groupings, can just summarise all polygons
                gdf_dissolved = gdf_clipped[["geometry"]].dissolve( aggfunc="sum").reset_index()
            
            # update to readable units by rounding
            if file_info_shp[file]["unit_type"].lower() == "length":
                gdf_dissolved['value'] = (gdf_dissolved.geometry.length).round(2)
            elif file_info_shp[file]["unit_type"].lower() == "area":
                gdf_dissolved['value'] = (gdf_dissolved.geometry.area/10000).round(2)

        # Update names of columns 
        gdf_dissolved['Name'] = pc
        if file_info_shp[file]["group_field"]:
            gdf_dissolved['groupColumnValue'] = gdf_dissolved[file_info_shp[file]["group_field"]]
            # If required, update values in column
            if file_info_shp[file]["groupColumnValue"]:
                if file == "Land Cover":
                    gdf_dissolved['groupColumnValue'] = gdf_dissolved['groupColumnValue'].astype(str).map(file_info_shp[file]["groupColumnValue"])
                else:
                    gdf_dissolved['groupColumnValue'] = gdf_dissolved['groupColumnValue'].map(file_info_shp[file]["groupColumnValue"])
                
            
        else:
            gdf_dissolved['groupColumnValue'] = "Total"

        
        # else:
        #     gdf_dissolved['groupColumnValue'] = "Total"
            
        gdf_dissolved['groupColumnName'] = file_info_shp[file]["groupColumnName"]
        gdf_dissolved['mapGroup'] = file_info_shp[file]["group"]
        gdf_dissolved['unitName'] = file_info_shp[file]["unit_type"]
        gdf_dissolved['unit'] = file_info_shp[file]["unit"]
        gdf_dissolved['datasetName'] = file
        gdf_dissolved['source'] = file_info_shp[file]["source"]

        gdf_dissolved = gdf_dissolved[['Name', "value","mapGroup","groupColumnName","groupColumnValue","unitName","unit","datasetName", "source"]]

        df_local_shp = pd.concat([df_local_shp,gdf_dissolved ])
    # Output as individual Csv's so can be saved
    df_local_shp.to_csv(f"{cwd}/data/clippedData/{file}_v1.csv", index = False)    
    df_all_shp = pd.concat([df_all_shp,df_local_shp ])
    

Landscape Character Areas
Bedrock
Faultlines
Flood Warning Areas
Kennet and Avon Canals Planning Buffer
Main Rivers
Water Framework Directive River, Canal, and Surface Water Transfer Water Bodies
National Forest Inventory
Greenspace
Priority Habitat Inventory


c:\Users\LaraS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


WSBRC Priority Habitats
Chalk Streams


c:\Users\LaraS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(


Ancient Woodland Inventory
Landscape Biodiversity Areas
National Cycle Network Route
Public Rights of Way


c:\Users\LaraS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(


CRoW_S15 Land All Types
CRoW Access Land
Country Parks
WWT reserves
Priority Rivers
Sites of Special Scientific Interest
Special Areas of Conservation
Special Protection Areas
National Trust Land Always Open
National Trust Land Limited Access


c:\Users\LaraS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(


RSPB Reserves
Woodland Trust Sites
Listed Buildings
Scheduled Monuments
Registered Parks and Gardens
World Heritage Sites
Battlefields
Conservation areas
Environmental Stewardship Scheme Agreements
Countryside Stewardship Scheme
Agricultural Land Classification
Forestry Commission
Protected Roadside Verges
County Wildlife Sites
National Nature Reserves
Local Nature Reserves


In [6]:
### To do
## List
# LNR (local nature reserve)
# NNR (national nature reserve)
# RPG ()
# SAM
# WHS
# Forestry commission sites
# wiltshire council sites
# MOD SPTA 
# Living churchyard

## Files
#     # "Traditional Orchards":{
#     #     "file_name":"G:\NE_OpenData\PHI\NE_Priority_Habitat_Inventory_v3-11-22_Wilts.shp",
#     #     "source":"NE",
#     #     "group":"Priority Habitat",
#     #                                         "group_field":False,
#     #                                         "unit":"HECTARES",
#     #                                         "unit_type":"AREA",
#     #                                         "area_col_name":"AREA"},  



#      "NE_Woodpasture_Parkland_Councils":{"group":"Priority Habitat",
#                                             "group_field":False,
#                                             "unit":"HECTARES",
#                                             "unit_type":"AREA",
#                                             "area_col_name":"AREA"},
#     # "NFI_AT":{"group":"TBC",
#     #                                         "group_field":"IFT_IOA",
#     #                                         "unit":"HECTARES",
#     #                                         "unit_type":"AREA",
#     #                                         "area_col_name":"Area_ha"},
#     "Protected_Landscapes_Councils":{"group":"Protected Landscape",
#                                             "group_field":False,
#                                             "unit":"HECTARES",
#                                             "unit_type":"AREA",
#                                             "area_col_name":"AREA"},  
#      "RCN_Councils":{"group":"Access and Routeways",
#                                             "group_field":False,
#                                             "unit":"Metres",
#                                             "unit_type":"LENGTH",
#                                             "area_col_name":"LENGTH"},       
# #     "RIGS_Councils":{"group":"Geology",
# #                                             "group_field":False,
# #                                             "unit":"Metres",
# #                                             "unit_type":"LENGTH",
# #                                             "area_col_name":"LENGTH"},
  
# 
#     # "Hedgerows_Councils":{"group":"Priority Habitat",
    #                                         "group_field":False,
    #                                         "unit":"Metres",
    #                                         "unit_type":"LENGTH",
    #                                         "area_col_name":"LENGTH"},
    # "HE_WHS_Count_Councils":{"group":"TBC",
    #                                         "group_field":False,
    #                                         "unit":"COUNT",
    #                                         "unit_type":"COUNT",
    #                                         "area_col_name":"COUNT",},   
    #  # "HLC_Councils":{"group":"TBC",
    #  #                                        "group_field":False,
    #  #                                        "unit":"HECTARES",
    #  #                                        "unit_type":"AREA",
    #  #                                        "area_col_name":"AREA",},
    # "LCA_Councils":{"group":"Landscape Character",
    #                                         "group_field":"Wc_lct",
    #                                         "unit":"HECTARES",
    #                                         "unit_type":"AREA",
    #                                         "area_col_name":"AREA",},       
    # "NCN_Councils":{"group":"Access and Routeways",
    #                                         "group_field":False,
    #                                         "unit":"Metres",
    #                                         "unit_type":"LENGTH",
    #                                         "area_col_name":"LENGTH"}, 